## Set up

### libraries

In [123]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict


In [124]:
# ==================== CONFIGURATION MAPPING ====================
# Easy customization: Change what each number in the config represents
# Position 3 (3rd dot-separated value): Turbulence models or other classifications
# Position 4 (4th dot-separated value): Grid/Adaptation status or other classifications

CONFIG_MAPPING = {
    'position_3': {
        '1': 'SST',
        '2': 'RNG',
        '3': 'RSM'
        # Add more as needed: '4': 'NewModel', '5': 'AnotherModel', etc.
    },
    'position_4': {
        '1': 'Baseline',
        '2': 'Baseline',
        '3': 'Adapted',
        '4': 'Grid_Fins'
        # Add more as needed: '5': 'CustomConfig', '6': 'AnotherStatus', etc.
    }
}

print("✓ Configuration mapping loaded")
print(f"  Position 3 options: {list(CONFIG_MAPPING['position_3'].values())}")
print(f"  Position 4 options: {list(CONFIG_MAPPING['position_4'].values())}")


✓ Configuration mapping loaded
  Position 3 options: ['SST', 'RNG', 'RSM']
  Position 4 options: ['Baseline', 'Baseline', 'Adapted', 'Grid_Fins']


### Data Extraction

In [125]:
def load_lift_drag_data(root_dir):
    data_by_config_aoa = defaultdict(lambda: {'lift': [], 'drag': [], 'turbulence_model': '', 'adapted': '', 'grid_fins': ''})
    
    for dirpath, _, filenames in os.walk(root_dir):
        # Find lift and drag files (with pattern like lift_force_AoA_X_3_1.txt)
        lift_files = [f for f in filenames if 'lift_force' in f and f.endswith('.txt')]
        drag_files = [f for f in filenames if 'drag_force' in f and f.endswith('.txt')]
        
        if lift_files and drag_files:
            # Extract configuration (e.g., 4.3.1.2) from path
            parts = dirpath.split(os.sep)
            config = None
            aoa = None
            
            # Find config number (4.X.X.X format)
            for part in parts:
                if part[0].isdigit() and len(part) == 7 and part.count('.') == 3:
                    config = part
                    break
            
            # Find AoA value from folder name (e.g., AoA_10)
            for part in parts:
                if part.startswith('AoA_'):
                    aoa = part
                    break
            
            if config and aoa:
                # Extract configuration values
                config_parts = config.split('.')
                third_num = config_parts[2] if len(config_parts) > 2 else "unknown"
                last_num = config_parts[3] if len(config_parts) > 3 else "unknown"
                
                # Get turbulence model from CONFIG_MAPPING
                turbulence_model = CONFIG_MAPPING['position_3'].get(third_num, "Unknown")
                
                # Get status from CONFIG_MAPPING for position_4
                # Special handling: position_4 uses custom logic for adapted/grid_fins
                # Check the mapping and set flags accordingly
                position_4_value = CONFIG_MAPPING['position_4'].get(last_num, "Unknown")
                
                if position_4_value == "Baseline":
                    adapted = "No"
                    grid_fins = "No"
                elif position_4_value == "Adapted":
                    adapted = "Yes"
                    grid_fins = "No"
                elif position_4_value == "Grid_Fins":
                    adapted = "No"
                    grid_fins = "Yes"
                else:
                    adapted = "Unknown"
                    grid_fins = "Unknown"
                
                # Read all lift files
                for lift_file in lift_files:
                    lift_path = os.path.join(dirpath, lift_file)
                    with open(lift_path) as f:
                        lift_data = []
                        for line in f:
                            line = line.strip()
                            # Skip header lines (lines with quotes or non-numeric content)
                            if not line or '"' in line or '(' in line:
                                continue
                            # Split by whitespace and take the second column
                            parts_line = line.split()
                            if len(parts_line) >= 2:
                                try:
                                    value = float(parts_line[1])
                                    lift_data.append(value)
                                except ValueError:
                                    continue
                        data_by_config_aoa[(config, aoa)]['lift'].extend(lift_data)
                
                # Read all drag files
                for drag_file in drag_files:
                    drag_path = os.path.join(dirpath, drag_file)
                    with open(drag_path) as f:
                        drag_data = []
                        for line in f:
                            line = line.strip()
                            # Skip header lines (lines with quotes or non-numeric content)
                            if not line or '"' in line or '(' in line:
                                continue
                            # Split by whitespace and take the second column
                            parts_line = line.split()
                            if len(parts_line) >= 2:
                                try:
                                    value = float(parts_line[1])
                                    drag_data.append(value)
                                except ValueError:
                                    continue
                        data_by_config_aoa[(config, aoa)]['drag'].extend(drag_data)
                
                # Store metadata
                data_by_config_aoa[(config, aoa)]['turbulence_model'] = turbulence_model
                data_by_config_aoa[(config, aoa)]['adapted'] = adapted
                data_by_config_aoa[(config, aoa)]['grid_fins'] = grid_fins
    
    return data_by_config_aoa


In [126]:
def compute_statistics(data):
    mean_val = np.mean(data)
    std_dev = np.std(data)
    return mean_val, std_dev

## Enter Values Here

In [127]:
# ==================== CONFIGURATION ====================
# User parameters - modify these as needed

base_path = r"C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3"
output_dir = r"C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3"
num_iterations = 200  # Number of latest iterations to use for statistics

# ==================== PROCESSING ====================

# Load all data
all_data = load_lift_drag_data(base_path)

# Create output directories
os.makedirs(output_dir, exist_ok=True)
raw_data_dir = os.path.join(output_dir, "raw_data")
os.makedirs(raw_data_dir, exist_ok=True)

# Sort AoA values numerically (5, 10, 15, 20)
def extract_aoa_number(aoa_str):
    return int(aoa_str.split('_')[1])

# Export each dataset with a unique name based on config and AOA
for (config, aoa), data in all_data.items():
    filename_base = f"{config}_{aoa}"
    
    # Export lift data to raw_data folder
    lift_output = os.path.join(raw_data_dir, f"{filename_base}_lift.txt")
    with open(lift_output, 'w') as f:
        for value in data['lift']:
            f.write(f"{value}\n")
    
    # Export drag data to raw_data folder
    drag_output = os.path.join(raw_data_dir, f"{filename_base}_drag.txt")
    with open(drag_output, 'w') as f:
        for value in data['drag']:
            f.write(f"{value}\n")
    
    print(f"Exported: {filename_base}")

# Create summary statistics file using specified number of iterations
summary_file = os.path.join(output_dir, "SUMMARY_Statistics.txt")

with open(summary_file, 'w') as f:
    f.write("=" * 100 + "\n")
    f.write(f"SUMMARY STATISTICS - Last {num_iterations} Iterations\n")
    f.write("=" * 100 + "\n\n")
    
    # Sort by config and then by AoA numerically
    sorted_data = sorted(all_data.items(), key=lambda x: (x[0][0], extract_aoa_number(x[0][1])))
    
    for (config, aoa), data in sorted_data:
        # Get last N iterations (or all if less than N)
        lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
        drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
        
        # Calculate statistics
        lift_mean = np.mean(lift_last_n) if lift_last_n else 0
        lift_std = np.std(lift_last_n) if lift_last_n else 0
        drag_mean = np.mean(drag_last_n) if drag_last_n else 0
        drag_std = np.std(drag_last_n) if drag_last_n else 0
        
        # Write to file
        f.write(f"Configuration: {config}\n")
        f.write(f"Turbulence Model: {data['turbulence_model']}\n")
        f.write(f"Adapted: {data['adapted']}\n")
        f.write(f"Grid Fins: {data['grid_fins']}\n")
        f.write(f"Angle of Attack: {aoa}\n")
        f.write(f"Iterations used: {len(lift_last_n)}\n\n")
        f.write(f"LIFT:\n")
        f.write(f"  Average: {lift_mean:12.6f}\n")
        f.write(f"  Std Dev: {lift_std:12.6f}\n\n")
        f.write(f"DRAG:\n")
        f.write(f"  Average: {drag_mean:12.6f}\n")
        f.write(f"  Std Dev: {drag_std:12.6f}\n")
        f.write("-" * 100 + "\n\n")

print(f"\n✓ Raw data exported to: {raw_data_dir}")
print(f"✓ Summary statistics saved to: {summary_file}")


Exported: 4.3.1.2_AoA_10
Exported: 4.3.1.2_AoA_15
Exported: 4.3.1.2_AoA_20
Exported: 4.3.1.2_AoA_5
Exported: 4.3.1.3_AoA_10
Exported: 4.3.1.3_AoA_15
Exported: 4.3.1.3_AoA_20
Exported: 4.3.1.3_AoA_5
Exported: 4.3.1.4_AoA_10
Exported: 4.3.1.4_AoA_15
Exported: 4.3.1.4_AoA_20
Exported: 4.3.1.4_AoA_5
Exported: 4.3.2.1_AoA_10
Exported: 4.3.2.1_AoA_15
Exported: 4.3.2.1_AoA_20
Exported: 4.3.2.1_AoA_5
Exported: 4.3.3.1_AoA_10
Exported: 4.3.3.1_AoA_15
Exported: 4.3.3.1_AoA_20
Exported: 4.3.3.1_AoA_5

✓ Raw data exported to: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3\raw_data
✓ Summary statistics saved to: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3\SUMMARY_Statistics.txt


In [128]:
# ==================== CREATE BAR GRAPHS ====================

graphs_dir = os.path.join(output_dir, "graphs")
os.makedirs(graphs_dir, exist_ok=True)

# Group data by AoA and turbulence model
# Create dynamic dictionary structure based on CONFIG_MAPPING
model_structure = {model: {'lift': [], 'drag': []} for model in CONFIG_MAPPING['position_3'].values()}
aoa_groups = defaultdict(lambda: {model: {'lift': [], 'drag': []} for model in CONFIG_MAPPING['position_3'].values()})

for (config, aoa), data in all_data.items():
    # Skip if adapted=Yes or grid_fins=Yes
    if data['adapted'] == "Yes" or data['grid_fins'] == "Yes":
        continue
    
    turbulence_model = data['turbulence_model']
    
    # Get last N iterations for graphing
    lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
    drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
    
    # Calculate means
    lift_mean = np.mean(lift_last_n) if lift_last_n else 0
    drag_mean = np.mean(drag_last_n) if drag_last_n else 0
    
    if turbulence_model in aoa_groups[aoa]:
        aoa_groups[aoa][turbulence_model]['lift'].append(lift_mean)
        aoa_groups[aoa][turbulence_model]['drag'].append(drag_mean)

# Create bar graphs for each AoA
for aoa in sorted(aoa_groups.keys(), key=extract_aoa_number):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Prepare data for plotting
    models = list(CONFIG_MAPPING['position_3'].values())
    lift_means = []
    drag_means = []
    
    for model in models:
        if aoa_groups[aoa][model]['lift']:
            lift_means.append(np.mean(aoa_groups[aoa][model]['lift']))
        else:
            lift_means.append(0)
        
        if aoa_groups[aoa][model]['drag']:
            drag_means.append(np.mean(aoa_groups[aoa][model]['drag']))
        else:
            drag_means.append(0)
    
    # Create bar charts
    x = np.arange(len(models))
    width = 0.6
    
    ax1.bar(x, lift_means, width, color=['#1f77b4', '#ff7f0e', '#2ca02c'], edgecolor='black', linewidth=1.5)
    ax1.set_title(f'Lift Comparison - {aoa}', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Lift Mean Value', fontsize=12)
    ax1.set_xticks(x)
    ax1.set_xticklabels(models, fontsize=11)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(lift_means):
        ax1.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    
    ax2.bar(x, drag_means, width, color=['#1f77b4', '#ff7f0e', '#2ca02c'], edgecolor='black', linewidth=1.5)
    ax2.set_title(f'Drag Comparison - {aoa}', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Drag Mean Value', fontsize=12)
    ax2.set_xticks(x)
    ax2.set_xticklabels(models, fontsize=11)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(drag_means):
        ax2.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    
    # Save graph
    aoa_num = extract_aoa_number(aoa)
    graph_file = os.path.join(graphs_dir, f"turbulence_comparison_AoA_{aoa_num}.png")
    plt.savefig(graph_file, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Created bar graph for {aoa}: {graph_file}")

print(f"\n✓ All bar graphs saved to: {graphs_dir}")
print(f"✓ Note: Graphs exclude data with Adapted=Yes or Grid Fins=Yes")


Created bar graph for AoA_5: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3\graphs\turbulence_comparison_AoA_5.png
Created bar graph for AoA_10: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3\graphs\turbulence_comparison_AoA_10.png
Created bar graph for AoA_10: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3\graphs\turbulence_comparison_AoA_10.png
Created bar graph for AoA_15: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3\graphs\turbulence_comparison_AoA_15.png
Created bar graph for AoA_15: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3\graphs\turbulence_comparison_AoA_15.png
Created bar graph for AoA_20: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3\graphs\turbulence_comparison_AoA_20.png

✓ All bar graphs saved to: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3\graphs
✓ Note: Graphs exclude data with Adapted=Yes or Grid Fins=Yes
Created bar graph for AoA_20: C:\User

In [129]:
# ==================== CREATE BAR GRAPHS - GRID FINS ON ====================

# Group data by AoA (only Grid Fins = Yes)
grid_fins_data = defaultdict(lambda: {'lift': [], 'drag': []})

for (config, aoa), data in all_data.items():
    # Only include if grid_fins=Yes (last number = 4)
    if data['grid_fins'] != "Yes":
        continue
    
    # Get last N iterations for graphing
    lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
    drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
    
    # Calculate means
    lift_mean = np.mean(lift_last_n) if lift_last_n else 0
    drag_mean = np.mean(drag_last_n) if drag_last_n else 0
    
    grid_fins_data[aoa]['lift'].append(lift_mean)
    grid_fins_data[aoa]['drag'].append(drag_mean)

# Create one large bar graph comparing all AoAs (Grid Fins)
if grid_fins_data:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Prepare data for plotting
    aoas = sorted(grid_fins_data.keys(), key=extract_aoa_number)
    lift_means = []
    drag_means = []
    
    for aoa in aoas:
        if grid_fins_data[aoa]['lift']:
            lift_means.append(np.mean(grid_fins_data[aoa]['lift']))
        else:
            lift_means.append(0)
        
        if grid_fins_data[aoa]['drag']:
            drag_means.append(np.mean(grid_fins_data[aoa]['drag']))
        else:
            drag_means.append(0)
    
    # Create bar charts
    x = np.arange(len(aoas))
    width = 0.6
    
    ax1.bar(x, lift_means, width, color='#1f77b4', edgecolor='black', linewidth=1.5)
    ax1.set_title('Lift Comparison - Grid Fins ON (All AoA)', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Lift Mean Value', fontsize=12)
    ax1.set_xlabel('Angle of Attack', fontsize=12)
    ax1.set_xticks(x)
    ax1.set_xticklabels(aoas, fontsize=11)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(lift_means):
        ax1.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    
    ax2.bar(x, drag_means, width, color='#ff7f0e', edgecolor='black', linewidth=1.5)
    ax2.set_title('Drag Comparison - Grid Fins ON (All AoA)', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Drag Mean Value', fontsize=12)
    ax2.set_xlabel('Angle of Attack', fontsize=12)
    ax2.set_xticks(x)
    ax2.set_xticklabels(aoas, fontsize=11)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(drag_means):
        ax2.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    
    # Save graph
    graph_file = os.path.join(graphs_dir, f"grid_fins_all_aoa_comparison.png")
    plt.savefig(graph_file, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Created grid fins comparison graph: {graph_file}")

print(f"✓ Grid fins bar graph saved to: {graphs_dir}")

# ==================== CREATE BAR GRAPHS - ADAPTED ====================

# Group data by AoA (only Adapted = Yes)
adapted_data = defaultdict(lambda: {'lift': [], 'drag': []})

for (config, aoa), data in all_data.items():
    # Only include if adapted=Yes (last number = 3)
    if data['adapted'] != "Yes":
        continue
    
    # Get last N iterations for graphing
    lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
    drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
    
    # Calculate means
    lift_mean = np.mean(lift_last_n) if lift_last_n else 0
    drag_mean = np.mean(drag_last_n) if drag_last_n else 0
    
    adapted_data[aoa]['lift'].append(lift_mean)
    adapted_data[aoa]['drag'].append(drag_mean)

# Create one large bar graph comparing all AoAs (Adapted)
if adapted_data:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Prepare data for plotting
    aoas = sorted(adapted_data.keys(), key=extract_aoa_number)
    lift_means = []
    drag_means = []
    
    for aoa in aoas:
        if adapted_data[aoa]['lift']:
            lift_means.append(np.mean(adapted_data[aoa]['lift']))
        else:
            lift_means.append(0)
        
        if adapted_data[aoa]['drag']:
            drag_means.append(np.mean(adapted_data[aoa]['drag']))
        else:
            drag_means.append(0)
    
    # Create bar charts
    x = np.arange(len(aoas))
    width = 0.6
    
    ax1.bar(x, lift_means, width, color='#2ca02c', edgecolor='black', linewidth=1.5)
    ax1.set_title('Lift Comparison - Adapted (All AoA)', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Lift Mean Value', fontsize=12)
    ax1.set_xlabel('Angle of Attack', fontsize=12)
    ax1.set_xticks(x)
    ax1.set_xticklabels(aoas, fontsize=11)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(lift_means):
        ax1.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    
    ax2.bar(x, drag_means, width, color='#d62728', edgecolor='black', linewidth=1.5)
    ax2.set_title('Drag Comparison - Adapted (All AoA)', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Drag Mean Value', fontsize=12)
    ax2.set_xlabel('Angle of Attack', fontsize=12)
    ax2.set_xticks(x)
    ax2.set_xticklabels(aoas, fontsize=11)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate(drag_means):
        ax2.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    
    # Save graph
    graph_file = os.path.join(graphs_dir, f"adapted_all_aoa_comparison.png")
    plt.savefig(graph_file, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Created adapted comparison graph: {graph_file}")

print(f"✓ Adapted bar graph saved to: {graphs_dir}")


Created grid fins comparison graph: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3\graphs\grid_fins_all_aoa_comparison.png
✓ Grid fins bar graph saved to: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3\graphs
Created adapted comparison graph: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3\graphs\adapted_all_aoa_comparison.png
✓ Adapted bar graph saved to: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3\graphs
Created adapted comparison graph: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3\graphs\adapted_all_aoa_comparison.png
✓ Adapted bar graph saved to: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3\graphs


In [130]:
# ==================== CREATE EXCEL TABLES ====================

from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# Create Excel file with all data
excel_file = os.path.join(output_dir, "SUMMARY_Statistics.xlsx")

# Recreate config_groups for reference
config_groups = defaultdict(lambda: {'lift': {}, 'drag': {}})
for (config, aoa), data in all_data.items():
    config_parts = config.split('.')
    third_num = config_parts[2] if len(config_parts) > 2 else "unknown"
    
    lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
    drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
    
    config_groups[third_num]['lift'][aoa] = lift_last_n
    config_groups[third_num]['drag'][aoa] = drag_last_n

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    # Create a summary sheet with all configurations
    summary_data = []
    
    # Sort by config and then by AoA numerically
    sorted_data = sorted(all_data.items(), key=lambda x: (x[0][0], extract_aoa_number(x[0][1])))
    
    for (config, aoa), data in sorted_data:
        # Get last N iterations
        lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
        drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
        
        # Calculate statistics
        lift_mean = np.mean(lift_last_n) if lift_last_n else 0
        lift_std = np.std(lift_last_n) if lift_last_n else 0
        lift_cov = (lift_std / lift_mean * 100) if lift_mean != 0 else 0
        drag_mean = np.mean(drag_last_n) if drag_last_n else 0
        drag_std = np.std(drag_last_n) if drag_last_n else 0
        drag_cov = (drag_std / drag_mean * 100) if drag_mean != 0 else 0
        
        summary_data.append({
            'Configuration': config,
            'Turbulence Model': data['turbulence_model'],
            'Adapted': data['adapted'],
            'Grid Fins': data['grid_fins'],
            'Angle of Attack': aoa,
            'Iterations Used': len(lift_last_n),
            'Lift Mean (N)': f"{lift_mean:.3f}",
            'Lift COV (%)': f"{lift_cov:.2f}",
            'Drag Mean (N)': f"{drag_mean:.3f}",
            'Drag COV (%)': f"{drag_cov:.2f}"
        })
    
    # Write summary to main sheet
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    
    # Create consolidated Turbulence Models sheet
    turbulence_models_data = []
    
    for third_num in sorted(config_groups.keys()):
        # Sort AoA numerically within each config type
        sorted_aoas = sorted(config_groups[third_num]['lift'].keys(), key=extract_aoa_number)
        
        for aoa in sorted_aoas:
            lift_data = config_groups[third_num]['lift'][aoa]
            drag_data = config_groups[third_num]['drag'][aoa]
            
            # Get turbulence model info (should be same for all in this group)
            turbulence_model = ""
            adapted = ""
            grid_fins = ""
            for (config, config_aoa), d in all_data.items():
                if config_aoa == aoa:
                    config_parts = config.split('.')
                    if config_parts[2] == third_num:
                        turbulence_model = d['turbulence_model']
                        adapted = d['adapted']
                        grid_fins = d['grid_fins']
                        break
            
            lift_mean_val = np.mean(lift_data)
            lift_std_val = np.std(lift_data)
            lift_cov_val = (lift_std_val / lift_mean_val * 100) if lift_mean_val != 0 else 0
            drag_mean_val = np.mean(drag_data)
            drag_std_val = np.std(drag_data)
            drag_cov_val = (drag_std_val / drag_mean_val * 100) if drag_mean_val != 0 else 0
            
            turbulence_models_data.append({
                'Turbulence Model': turbulence_model,
                'Angle of Attack': aoa,
                'Adapted': adapted,
                'Grid Fins': grid_fins,
                'Lift Mean (N)': f"{lift_mean_val:.3f}",
                'Lift COV (%)': f"{lift_cov_val:.2f}",
                'Drag Mean (N)': f"{drag_mean_val:.3f}",
                'Drag COV (%)': f"{drag_cov_val:.2f}",
                'Num Points': len(lift_data)
            })
    
    turbulence_models_df = pd.DataFrame(turbulence_models_data)
    turbulence_models_df.to_excel(writer, sheet_name='Turbulence_Models', index=False)

# Apply formatting to all sheets
from openpyxl import load_workbook
wb = load_workbook(excel_file)

# Define styles
header_font = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
header_fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
header_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

data_alignment = Alignment(horizontal='center', vertical='center')
border_style = Border(
    left=Side(style='thin', color='000000'),
    right=Side(style='thin', color='000000'),
    top=Side(style='thin', color='000000'),
    bottom=Side(style='thin', color='000000')
)

# Alternating row colors
row_fill_light = PatternFill(start_color='F2F2F2', end_color='F2F2F2', fill_type='solid')
row_fill_white = PatternFill(start_color='FFFFFF', end_color='FFFFFF', fill_type='solid')

for ws_name in wb.sheetnames:
    worksheet = wb[ws_name]
    
    # Autofit column widths
    for column in worksheet.columns:
        max_length = 0
        column_letter = get_column_letter(column[0].column)
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = min(max_length + 2, 50)
        worksheet.column_dimensions[column_letter].width = adjusted_width
        cell.alignment = header_alignment
    # Format header row
    for cell in worksheet[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = border_style
    
    # Format data rows with alternating colors and borders
    for row_idx, row in enumerate(worksheet.iter_rows(min_row=2), start=2):
        fill_color = row_fill_light if row_idx % 2 == 0 else row_fill_white
        for cell in row:
            cell.alignment = data_alignment
            cell.border = border_style
            cell.fill = fill_color
    
    # Set row height for header
    worksheet.row_dimensions[1].height = 30

wb.save(excel_file)

print(f"✓ Excel file created: {excel_file}")
print(f"✓ Sheet names: Summary, Turbulence_Models")
print(f"✓ Professional formatting applied: colored headers, borders, alternating rows")
print(f"✓ All lift and drag values displayed in Newtons (N)")

✓ Excel file created: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Test\2414_006_004.3\SUMMARY_Statistics.xlsx
✓ Sheet names: Summary, Turbulence_Models
✓ Professional formatting applied: colored headers, borders, alternating rows
✓ All lift and drag values displayed in Newtons (N)


In [131]:
# ==================== CREATE TURBULENCE MODEL COMPARISON TABLE ====================

import time

# Create a comparison table for turbulence models (baseline only: no adapted, no grid fins)
turbulence_comparison = []

# Get all unique AoAs
all_aoas = set()
for (config, aoa), data in all_data.items():
    if data['adapted'] == "No" and data['grid_fins'] == "No":
        all_aoas.add(aoa)

# Sort AoAs numerically
sorted_aoas = sorted(all_aoas, key=extract_aoa_number)

# For each AoA, get data for each turbulence model
for aoa in sorted_aoas:
    aoa_entry = {'Angle of Attack': aoa}
    
    # Collect lift and drag means and COV for each model
    lift_means = {}
    drag_means = {}
    lift_covs = {}
    drag_covs = {}
    
    for model in list(CONFIG_MAPPING['position_3'].values()):
        model_lift_values = []
        model_drag_values = []
        
        # Find all configs with this AOA and turbulence model (baseline only)
        for (config, config_aoa), data in all_data.items():
            if config_aoa == aoa and data['turbulence_model'] == model and data['adapted'] == "No" and data['grid_fins'] == "No":
                lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
                drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
                
                if lift_last_n:
                    model_lift_values.extend(lift_last_n)
                if drag_last_n:
                    model_drag_values.extend(drag_last_n)
        
        # Calculate means and COV for this model
        if model_lift_values:
            lift_means[model] = np.mean(model_lift_values)
            lift_std = np.std(model_lift_values)
            lift_covs[model] = (lift_std / lift_means[model] * 100) if lift_means[model] != 0 else 0
        else:
            lift_means[model] = None
            lift_covs[model] = None
        
        if model_drag_values:
            drag_means[model] = np.mean(model_drag_values)
            drag_std = np.std(model_drag_values)
            drag_covs[model] = (drag_std / drag_means[model] * 100) if drag_means[model] != 0 else 0
        else:
            drag_means[model] = None
            drag_covs[model] = None
    
    # Add lift values with COV and percent difference from SST
    lift_str = ""
    sst_lift = lift_means.get('SST')
    for model in list(CONFIG_MAPPING['position_3'].values()):
        if lift_means[model] is not None and lift_covs[model] is not None:
            if lift_str:
                lift_str += " | "
            # Calculate percent difference from SST
            if model != 'SST' and sst_lift is not None and sst_lift != 0:
                pct_diff = ((lift_means[model] - sst_lift) / sst_lift * 100)
                lift_str += f"{model}: {lift_means[model]:.3f} N ({lift_covs[model]:.2f}%, {pct_diff:+.2f}% vs SST)"
            else:
                lift_str += f"{model}: {lift_means[model]:.3f} N ({lift_covs[model]:.2f}%)"
        else:
            if lift_str:
                lift_str += " | "
            lift_str += f"{model}: N/A"
    aoa_entry['Lift (N)'] = lift_str
    
    # Add drag values with COV and percent difference from SST
    drag_str = ""
    sst_drag = drag_means.get('SST')
    for model in list(CONFIG_MAPPING['position_3'].values()):
        if drag_means[model] is not None and drag_covs[model] is not None:
            if drag_str:
                drag_str += " | "
            # Calculate percent difference from SST
            if model != 'SST' and sst_drag is not None and sst_drag != 0:
                pct_diff = ((drag_means[model] - sst_drag) / sst_drag * 100)
                drag_str += f"{model}: {drag_means[model]:.3f} N ({drag_covs[model]:.2f}%, {pct_diff:+.2f}% vs SST)"
            else:
                drag_str += f"{model}: {drag_means[model]:.3f} N ({drag_covs[model]:.2f}%)"
        else:
            if drag_str:
                drag_str += " | "
            drag_str += f"{model}: N/A"
    aoa_entry['Drag (N)'] = drag_str
    
    turbulence_comparison.append(aoa_entry)

# Add to existing Excel file
comparison_df = pd.DataFrame(turbulence_comparison)

# Write comparison sheet
with pd.ExcelWriter(excel_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    comparison_df.to_excel(writer, sheet_name='Turbulence_Model_Comparison', index=False)

# Apply formatting to Turbulence_Model_Comparison sheet
time.sleep(0.5)  # Brief delay to ensure file is fully released

from openpyxl.styles.colors import Color

wb = load_workbook(excel_file)
ws = wb['Turbulence_Model_Comparison']

# Define styles
header_font = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
header_fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
header_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
data_alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
border_style = Border(
    left=Side(style='thin', color='000000'),
    right=Side(style='thin', color='000000'),
    top=Side(style='thin', color='000000'),
    bottom=Side(style='thin', color='000000')
)
row_fill_light = PatternFill(start_color='F2F2F2', end_color='F2F2F2', fill_type='solid')
row_fill_white = PatternFill(start_color='FFFFFF', end_color='FFFFFF', fill_type='solid')

# Color coding for highlights
cov_color = Color(rgb='FF0000')  # Red for COV
pct_diff_color = Color(rgb='4169E1')  # Royal blue for percentage comparison

# Autofit column widths
for column in ws.columns:
    max_length = 0
    column_letter = get_column_letter(column[0].column)
    for cell in column:
        try:
            if len(str(cell.value)) > max_length:
                max_length = len(str(cell.value))
        except:
            pass
    adjusted_width = min(max_length + 2, 80)
    ws.column_dimensions[column_letter].width = adjusted_width

# Format header row with legend
for cell in ws[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = header_alignment
    cell.border = border_style

# Add legend row below header to explain format
ws.insert_rows(2)
legend_font = Font(name='Calibri', size=9, italic=True)
legend_fill = PatternFill(start_color='E0E0E0', end_color='E0E0E0', fill_type='solid')

ws['A2'] = 'Legend'
ws['B2'] = 'Format: Model: Mean N (COV%, %Diff vs SST) | Red=COV | Blue=%Difference'
ws['A2'].font = legend_font
ws['B2'].font = legend_font
ws['A2'].fill = legend_fill
ws['B2'].fill = legend_fill
ws['A2'].border = border_style
ws['B2'].border = border_style
ws['A2'].alignment = Alignment(horizontal='center', vertical='center')
ws['B2'].alignment = Alignment(horizontal='left', vertical='center')
ws.merge_cells('B2:C2')

# Format data rows with alternating colors, borders, and highlighting
for row_idx, row in enumerate(ws.iter_rows(min_row=3), start=3):
    fill_color = row_fill_light if row_idx % 2 == 1 else row_fill_white
    for cell_idx, cell in enumerate(row):
        cell.alignment = data_alignment
        cell.border = border_style
        cell.fill = fill_color
        
        # Apply color highlighting to Lift and Drag columns (columns B and C)
        if cell_idx in [1, 2] and cell.value:  # Columns B and C (Lift and Drag)
            cell_text = str(cell.value)
            from openpyxl.cell.text import InlineFont
            from openpyxl.cell.rich_text import TextBlock, CellRichText
            
            # Parse and create rich text with colored segments
            # Format is: "Model: Value N (COV%, +X% vs SST)" or "Model: Value N (COV%)"
            text_parts = []
            i = 0
            while i < len(cell_text):
                # Check for opening parenthesis
                if cell_text[i] == '(':
                    # Find the closing parenthesis
                    closing_paren = cell_text.find(')', i)
                    if closing_paren != -1:
                        segment = cell_text[i:closing_paren+1]
                        
                        # Check if this segment contains "vs SST"
                        if 'vs SST' in segment:
                            # This segment has both COV and percentage difference
                            # Split by comma to separate them
                            if ',' in segment:
                                parts = segment[1:-1].split(',', 1)  # Remove outer parentheses and split once
                                # First part is COV (e.g., "3.45%")
                                cov_part = f"({parts[0].strip()})"
                                text_parts.append(TextBlock(InlineFont(color=cov_color), cov_part))
                                # Second part is percentage diff (e.g., " +1.96% vs SST")
                                pct_part = f"({parts[1].strip()})"
                                text_parts.append(TextBlock(InlineFont(color=pct_diff_color), pct_part))
                            else:
                                # Shouldn't happen, but just in case
                                text_parts.append(TextBlock(InlineFont(color=pct_diff_color), segment))
                        elif '%' in segment:
                            # This is just COV percentage - color it red
                            text_parts.append(TextBlock(InlineFont(color=cov_color), segment))
                        else:
                            # No percentage, regular text
                            text_parts.append(TextBlock(InlineFont(), segment))
                        
                        i = closing_paren + 1
                        continue
                
                # Regular text (not in parentheses)
                j = i
                while j < len(cell_text) and cell_text[j] != '(':
                    j += 1
                if j > i:
                    text_parts.append(TextBlock(InlineFont(), cell_text[i:j]))
                i = j
            
            if text_parts:
                cell.value = CellRichText(*text_parts)

# Set row height for header and legend
ws.row_dimensions[1].height = 30
ws.row_dimensions[2].height = 25

wb.save(excel_file)
wb.close()

print(f"✓ Turbulence Model Comparison table added to Excel file")
print(f"✓ Format: Model: Mean (COV%) with percent difference from SST")
print(f"✓ Professional formatting applied to comparison sheet")
print(f"\nSample comparison:")
for row in turbulence_comparison:
    print(f"\n{row['Angle of Attack']}:")
    print(f"  Lift (N):  {row['Lift (N)']}")
    print(f"  Drag (N):  {row['Drag (N)']}")

✓ Turbulence Model Comparison table added to Excel file
✓ Format: Model: Mean (COV%) with percent difference from SST
✓ Professional formatting applied to comparison sheet

Sample comparison:

AoA_5:
  Lift (N):  SST: 62.980 N (0.00%) | RNG: 66.855 N (0.00%, +6.15% vs SST) | RSM: 64.587 N (0.00%, +2.55% vs SST)
  Drag (N):  SST: -3.503 N (-0.00%) | RNG: -3.780 N (-0.00%, +7.92% vs SST) | RSM: -3.700 N (-0.00%, +5.63% vs SST)

AoA_10:
  Lift (N):  SST: 98.612 N (0.00%) | RNG: 107.234 N (0.00%, +8.74% vs SST) | RSM: 101.459 N (0.00%, +2.89% vs SST)
  Drag (N):  SST: -13.603 N (-0.00%) | RNG: -14.999 N (-0.00%, +10.26% vs SST) | RSM: -14.164 N (-0.00%, +4.12% vs SST)

AoA_15:
  Lift (N):  SST: 71.565 N (2.72%) | RNG: 92.678 N (9.41%, +29.50% vs SST) | RSM: 85.675 N (10.41%, +19.72% vs SST)
  Drag (N):  SST: -4.760 N (-13.51%) | RNG: -16.133 N (-17.44%, +238.94% vs SST) | RSM: -7.026 N (-42.74%, +47.60% vs SST)

AoA_20:
  Lift (N):  SST: 76.173 N (0.19%) | RNG: 62.211 N (9.38%, -18.33% vs 